## CommonLit Readability Prize

This project predicts how difficult a passage of text is to read, as a number. The passages are excerpts from literature and informational texts, rated by teachers for use in grades 3 to 12. Automatic readability scores help teachers choose reading material that fits their students.

## Approach
1. Load the data and explore the target distribution and example excerpts
2. Convert text into numeric features with TF-IDF
3. Train a ridge regression and evaluate RMSE on a validation split
4. Generate predictions and create the submission file
5. Submit to Kaggle and record the score

In [1]:
import os
from getpass import getpass

os.environ["KAGGLE_API_TOKEN"] = getpass("Paste your Kaggle API token and press Enter: ")

!pip install -q -U kaggle
!kaggle competitions download -c commonlitreadabilityprize
!unzip -oq commonlitreadabilityprize.zip

Paste your Kaggle API token and press Enter: ··········
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 8.0 MB/s eta 0:00:00
100% 1.13M/1.13M [00:00<00:00, 2.21MB/s]



In [2]:
import pandas as pd
import numpy as np

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

In [3]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2834 entries, 0 to 2833
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              2834 non-null   object 
 1   url_legal       830 non-null    object 
 2   license         830 non-null    object 
 3   excerpt         2834 non-null   object 
 4   target          2834 non-null   float64
 5   standard_error  2834 non-null   float64
dtypes: float64(2), object(4)
memory usage: 133.0+ KB


In [4]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   id         7 non-null      object
 1   url_legal  3 non-null      object
 2   license    3 non-null      object
 3   excerpt    7 non-null      object
dtypes: object(4)
memory usage: 356.0+ bytes


In [5]:
train.sample(1)

,id,url_legal,license,excerpt,target,standard_error
923,ad84a0097,https://www.africanstorybook.org/,CC BY 4.0,One day Jackal felt sick. He tried drinking wa...,0.030959,0.475515


In [6]:
test.sample(1)

,id,url_legal,license,excerpt
2,0df072751,NaN,NaN,It was a bright and cheerful scene that greete...


In [7]:
print(train.shape, test.shape)

(2834, 6) (7, 4)


In [8]:
train.columns.tolist()

['id', 'url_legal', 'license', 'excerpt', 'target', 'standard_error']

In [9]:
train["target"].describe()

,target
count,2834.000000
mean,-0.959319
std,1.033579
min,-3.676268
25%,-1.690320
50%,-0.912190
75%,-0.202540
max,1.711390


In [10]:
train[["excerpt", "target"]].sort_values("target").iloc[[0, -1]]

,excerpt,target
1705,"The commutator is peculiar, consisting of only...",-3.676268
2829,When you think of dinosaurs and where they liv...,1.711390


In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), sublinear_tf=True)
x = vectorizer.fit_transform(train["excerpt"])
y = train["target"]

In [12]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.2, random_state=42)

model = Ridge(alpha=1.0)
model.fit(x_train, y_train)

pred = model.predict(x_val)
rmse = mean_squared_error(y_val, pred) ** 0.5
print("Validation RMSE:", round(rmse, 4))

Validation RMSE: 0.7093


In [13]:
model.fit(x, y)

x_test = vectorizer.transform(test["excerpt"])
pred_test = model.predict(x_test)

pd.DataFrame({"id": test["id"], "target": pred_test}).to_csv("submission.csv", index=False)

In [14]:
!kaggle competitions submit -c commonlitreadabilityprize -f submission.csv -m "TF-IDF + Ridge baseline"

100% 221/221 [00:00<00:00, 611B/s]
400 Client Error: Bad Request for url: https://api.kaggle.com/v1/competitions.CompetitionApiService/CreateSubmission


In [15]:
import pickle, os
pickle.dump(model, open("readability_model.pkl", "wb"))
pickle.dump(vectorizer, open("readability_vectorizer.pkl", "wb"))
print("model KB:", os.path.getsize("readability_model.pkl") // 1024)
print("vectorizer KB:", os.path.getsize("readability_vectorizer.pkl") // 1024)

model KB: 156
vectorizer KB: 745
